# Drowsy Driver Detection using ResNet18

## Project Objective

The objective of this project is to develop a driver drowsiness detection system using Deep Learning and Computer Vision techniques.

The system classifies images into two categories:

- Drowsy
- Not Drowsy

A pre-trained ResNet18 model is fine-tuned using transfer learning on a labeled dataset and later used for real-time detection through a webcam.

In [1]:
!pip install torch torchvision matplotlib scikit-learn

## Google Drive Integration

The dataset and trained model are stored in Google Drive.

This step mounts Google Drive inside the Google Colab environment to access project files.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls /content/drive/MyDrive

 11.gdoc
 1.gdoc
'Activit__2___Complexit__DFS_vs_BFS (1).gsheet'
 Activit__2___Complexit__DFS_vs_BFS.gsheet
"Applications de l'IA.gdoc"
' Architecture MVC en PHP.gdoc'
'archive (3).zip'
'ASSADIKI ZAKARIA'
'Assadiki-Zakaria CI-gitlab-python.gdoc'
'Assadiki-Zakaria crud-etudiant.gdoc'
'Assadiki-Zakaria Deploy-Kubernetes.gdoc'
 Assadiki_Zakaria_django1.gdoc
'Assadiki-Zakaria_Examen de Virtualisation et Conteneurisation.gdoc'
 Assadiki-Zakaria-laravel.gdoc
'Assadiki-Zakaria: php-poo.gdoc'
'ASSADIKI ZAKARIA TP2 (1).pdf'
'ASSADIKI ZAKARIA TP2.pdf'
' ASSADIKI-ZAKARIA  TP ROLES 2.gdoc'
'ASSADIKI_ZAKARIA_TP_ROLES privilege.gdoc'
 Atelier2-ASSADIKI_ZAKARIA.gdoc
'Atelier-5 ASSADIKI-ZAKARIA.gdoc'
'Atelier N°3-ASSADIKI-ZAKARIA  .gdoc'
'Atelier: Zakaria Assadiki.gdoc'
'Budget mensuel.gsheet'
'cahier de charge.gdoc'
 cc-gestion-projet.gdoc
 challengeHUB
'char concatenation(char ch1,char ch.txt'
 cloud_plan.gdoc
'Colab Notebooks'
'com.box.android-4.1 (1).568-www.APK4Fun.com.apk'
 com.box.android-4.1.5

## Environment Verification

This section verifies:

- GPU availability
- CUDA support
- Dataset classes
- Number of training, validation and testing samples

The project uses a Tesla T4 GPU provided by Google Colab for faster training.

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Using: cuda
Tesla T4


## Dataset Extraction

The compressed dataset is extracted from Google Drive into the Colab workspace.

The dataset contains two classes:

- drowsy
- notdrowsy

In [6]:
!mkdir -p /content/drowsy_dataset
!unzip -q "/content/drive/MyDrive/dataset.zip" -d /content/drowsy_dataset

## Data Preprocessing

Images are prepared before training using the following transformations:

- Resize to 224 × 224 pixels
- Conversion to tensor format
- Normalization using ImageNet statistics

The dataset is then divided into:

- Training Set
- Validation Set
- Test Set

In [7]:
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

DATASET_PATH = "/content/drowsy_dataset/Multi class/train"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

print("Classes:", dataset.classes)

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Classes: ['drowsy', 'notdrowsy']
Train: 46564
Validation: 9978
Test: 9979


## Transfer Learning with ResNet18

A pre-trained ResNet18 convolutional neural network is used.

Transfer learning is applied by replacing the original output layer with a new layer containing two classes:

- Drowsy
- Not Drowsy

This approach reduces training time and improves performance.

In [8]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 209MB/s]


Linear(in_features=512, out_features=2, bias=True)


## Training Configuration

Training parameters:

- Loss Function: Cross Entropy Loss
- Optimizer: Adam
- Learning Rate: 0.001
- Number of Epochs: 5

These parameters are used to fine-tune the ResNet18 model.

In [9]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5

## Model Training

The model is trained on the training dataset.

For each epoch:

1. Images are passed through the network.
2. Loss is computed.
3. Backpropagation is performed.
4. Model weights are updated.

Training accuracy is displayed after each epoch.

In [10]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.2f}%")

Epoch [1/5], Loss: 204.1353, Train Accuracy: 94.47%
Epoch [2/5], Loss: 90.1706, Train Accuracy: 97.66%
Epoch [3/5], Loss: 64.7192, Train Accuracy: 98.33%
Epoch [4/5], Loss: 54.2981, Train Accuracy: 98.71%
Epoch [5/5], Loss: 46.7879, Train Accuracy: 98.85%


## Validation Evaluation

The validation dataset is used to evaluate the model on unseen samples during development.

This provides an estimate of the model's generalization capability.

In [11]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total
print(f"Validation Accuracy: {val_acc:.2f}%")

Validation Accuracy: 98.78%


## Test Evaluation

The final evaluation is performed using the test dataset.

The resulting accuracy represents the final performance of the trained model on completely unseen data.

In [12]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 98.80%


## Model Export

The trained model is saved as:

`model.pt`

This file contains the learned weights and will be used later for real-time inference.

In [13]:
torch.save(model.state_dict(), "model.pt")
print("Model saved as model.pt")

Model saved as model.pt


## Backup to Google Drive

The trained model is copied to Google Drive to preserve it after the Colab session ends.

In [14]:
!cp model.pt "/content/drive/MyDrive/model.pt"
print("Model copied to Google Drive")

Model copied to Google Drive


In [15]:
!ls "/content/drive/MyDrive" | grep model.pt

model.pt


# Conclusion

A ResNet18 model was successfully fine-tuned using transfer learning for driver drowsiness detection.

Results obtained:

- Training Accuracy: 98.85%
- Validation Accuracy: 98.78%
- Test Accuracy: 98.80%

The trained model was exported as `model.pt` and later integrated into a real-time OpenCV application.

Although high accuracy was achieved on the dataset, real-world testing revealed limitations in model generalization due to dataset quality and label consistency. This project nevertheless demonstrates the complete deep learning workflow from data preparation to real-time deployment.